# Simulador de Voos com Scores da Gold

Este notebook integra duas camadas do projeto:

1. **consulta online de voos** via Google Flights/SerpApi;
2. **consulta da Gold já materializada** com os scores calculados a partir do histórico da ANAC.

O usuário informa:

- cidade de origem;
- cidade de destino;
- data pretendida da viagem.

A saída retorna as opções encontradas pela API, incluindo companhia aérea, tarifa, horários, duração e conexões, e faz o `JOIN` com a Gold por:

`data_voo × municipio_origem × municipio_destino × nome_empresa`

Além dos scores previamente materializados, o notebook calcula no momento da consulta um `score_preco_atual`, comparando a tarifa encontrada pela API com as referências históricas e com o preço estimado pelo modelo.

> A chave da SerpApi não fica gravada no notebook.


In [ ]:
# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import os
import re
import glob
import unicodedata
from getpass import getpass
from datetime import datetime, date

import httpx
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)
pl.Config.set_fmt_str_lengths(80)


In [ ]:
# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

# Colab Drive desativado — dados locais em data/SoR · SoT · Spec
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
# ============================================================
# 3. CONFIGURAÇÕES DO PROJETO
# ============================================================
# CONFIGURAÇÕES DO PROJETO (monorepo local)
# SoR ≈ Bronze · SoT ≈ Silver · Spec ≈ Gold
# ============================================================
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "data" / "SoR").exists():
    _data = _here / "data"
elif (_here.parent / "data" / "SoR").exists():
    _data = _here.parent / "data"
else:
    _data = _here / "data"

PASTA_PROJETO = str(_data)
PASTA_SOR = str(_data / "SoR")
PASTA_SOT = str(_data / "SoT")
PASTA_SPEC = str(_data / "Spec")
PATH_VOOS = (f"{PASTA_SOT}/SoT_historico_voos/*.parquet")

PADRAO_AEROPORTOS = (f"{PASTA_SOT}/SoT_aeroportos/aeroportos_*.parquet")


PADRAO_GOLD = fr"{PASTA_SPEC}/**/spec_modelos_risco_*.parquet"

URL_SERPAPI = "https://serpapi.com/search.json"


## 4. Chave da SerpApi

A chave não deve ser salva diretamente no código.

O notebook tenta, nesta ordem:

1. variável de ambiente `SERPAPI_API_KEY`;
2. Secrets do Google Colab;
3. entrada manual oculta.


In [ ]:
# ============================================================
# 4. OBTENÇÃO SEGURA DA API KEY
# ============================================================

def obter_serpapi_key():
    chave = os.getenv("SERPAPI_API_KEY")

    if chave:
        return chave

    try:
        from google.colab import userdata
        chave = userdata.get("SERPAPI_API_KEY")
        if chave:
            return chave
    except Exception:
        pass

    chave = getpass("Informe sua SERPAPI_API_KEY: ").strip()

    if not chave:
        raise ValueError("SERPAPI_API_KEY não informada.")

    return chave


API_KEY = obter_serpapi_key()

print("API key carregada com segurança.")


<!-- chave movida para apps/api/.env (SERPAPI_API_KEY) — não versionar -->


## 5. Funções de normalização

As fontes podem representar a mesma cidade ou companhia de formas diferentes, por exemplo:

- `São Paulo` e `Sao Paulo`;
- `LATAM Airlines` e `LATAM`;
- `Azul Linhas Aéreas` e `Azul`.

As funções abaixo criam chaves técnicas apenas para o `JOIN`. Os nomes originais continuam preservados na saída.


In [ ]:
# ============================================================
# 5.1 NORMALIZAÇÃO DE TEXTO
# ============================================================

def normalizar_texto(valor):
    if valor is None:
        return None

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    texto = re.sub(r"\\s+", " ", texto).strip()

    return texto


def normalizar_companhia(valor):
    texto = normalizar_texto(valor)

    if not texto:
        return None

    if "azul" in texto:
        return "azul"

    if "latam" in texto or "tam linhas" in texto:
        return "latam"

    if re.search(r"(^| )gol($| )", texto) or "gol linhas" in texto:
        return "gol"

    if "voepass" in texto or "passaredo" in texto:
        return "voepass"

    if "avianca" in texto:
        return "avianca"

    remover = [
        "linhas aereas",
        "transportes aereos",
        "companhia aerea",
        "airlines",
        "airways",
        "s a",
        "sa",
    ]

    for trecho in remover:
        texto = texto.replace(trecho, " ")

    texto = re.sub(r"\\s+", " ", texto).strip()

    return texto


In [ ]:
# ============================================================
# 5.2 EXPRESSÕES POLARS PARA NORMALIZAÇÃO VETORIZADA
# ============================================================

def expr_texto_normalizado(coluna):
    expr = (
        pl.col(coluna)
        .cast(pl.String)
        .str.to_lowercase()
        .str.strip_chars()
    )

    substituicoes = [
        (r"[áàãâä]", "a"),
        (r"[éèêë]", "e"),
        (r"[íìîï]", "i"),
        (r"[óòõôö]", "o"),
        (r"[úùûü]", "u"),
        (r"ç", "c"),
    ]

    for padrao, substituto in substituicoes:
        expr = expr.str.replace_all(padrao, substituto)

    return (
        expr
        .str.replace_all(r"[^a-z0-9]+", " ")
        .str.replace_all(r"\\s+", " ")
        .str.strip_chars()
    )


def expr_companhia_normalizada(coluna):
    base = expr_texto_normalizado(coluna)

    return (
        pl.when(base.str.contains("azul"))
        .then(pl.lit("azul"))
        .when(
            base.str.contains("latam")
            | base.str.contains("tam linhas")
        )
        .then(pl.lit("latam"))
        .when(
            base.str.contains(r"(^| )gol($| )")
            | base.str.contains("gol linhas")
        )
        .then(pl.lit("gol"))
        .when(
            base.str.contains("voepass")
            | base.str.contains("passaredo")
        )
        .then(pl.lit("voepass"))
        .when(base.str.contains("avianca"))
        .then(pl.lit("avianca"))
        .otherwise(base)
    )


## 6. Leitura da Gold

O notebook localiza automaticamente a versão mais recente de `spec_modelos_risco_*.parquet`.

A Gold não precisa ser carregada inteira para cada consulta. O `LazyFrame` permite filtrar primeiro pela data da viagem e somente depois coletar as linhas necessárias.


In [ ]:
# ============================================================
# 6.1 LOCALIZAÇÃO DA GOLD MAIS RECENTE
# ============================================================

arquivos_gold = glob.glob(
    PADRAO_GOLD,
    recursive=True,
)

if not arquivos_gold:
    raise FileNotFoundError(
        "Nenhuma Gold spec_modelos_risco_*.parquet foi encontrada em "
        f"{PASTA_SPEC}."
    )

ARQUIVO_GOLD = max(
    arquivos_gold,
    key=os.path.getmtime,
)

print("Gold selecionada:")
print(ARQUIVO_GOLD)


In [ ]:
# ============================================================
# 6.2 SCAN DA GOLD
# ============================================================

gold_lazy = pl.scan_parquet(ARQUIVO_GOLD)

schema_gold = gold_lazy.collect_schema()

print("Colunas da Gold:")
print(schema_gold.names())

intervalo_gold = (
    gold_lazy
    .select([
        pl.col("data_voo").min().alias("data_min"),
        pl.col("data_voo").max().alias("data_max"),
    ])
    .collect()
)

DATA_MIN_GOLD = intervalo_gold["data_min"][0]
DATA_MAX_GOLD = intervalo_gold["data_max"][0]

print(
    f"Período disponível na Gold: "
    f"{DATA_MIN_GOLD} até {DATA_MAX_GOLD}"
)


## 7. Dimensão município × aeroporto

O arquivo de origens/destinos já usado no projeto é convertido em uma dimensão:

`municipio → IATA`

Isso permite que o usuário digite o nome da cidade. A função também aceita diretamente um código IATA de três letras.


In [ ]:
# ============================================================
# 7.1 LEITURA DA SILVER DE AEROPORTOS
# ============================================================

arquivos_aeroportos = glob.glob(
    PADRAO_AEROPORTOS
)

if not arquivos_aeroportos:
    raise FileNotFoundError(
        f"Nenhum arquivo encontrado em {PADRAO_AEROPORTOS}"
    )

ARQUIVO_AEROPORTOS = max(
    arquivos_aeroportos,
    key=os.path.getmtime
)

df_aeroportos = (
    pl.read_parquet(ARQUIVO_AEROPORTOS)
    .select([
        "codigo_icao",
        "codigo_iata",
        "nome_aeroporto"
    ])
    .filter(
        pl.col("codigo_icao").is_not_null()
        & pl.col("codigo_iata").is_not_null()
    )
    .with_columns([
        pl.col("codigo_icao")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),

        pl.col("codigo_iata")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .unique()
)

df_aeroportos.head()

In [ ]:
# ============================================================
# 7.2 MUNICÍPIO ANAC + ICAO
# ============================================================

df_aeroportos_anac_origem = (
    pl.scan_parquet(PATH_VOOS)
    .select([
        pl.col("municipio_origem")
        .cast(pl.String)
        .str.strip_chars()
        .alias("municipio"),

        pl.col("icao_aerodromo_origem")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .alias("codigo_icao")
    ])
)

df_aeroportos_anac_destino = (
    pl.scan_parquet(PATH_VOOS)
    .select([
        pl.col("municipio_destino")
        .cast(pl.String)
        .str.strip_chars()
        .alias("municipio"),

        pl.col("icao_aerodromo_destino")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .alias("codigo_icao")
    ])
)

df_aeroportos_anac = (
    pl.concat([
        df_aeroportos_anac_origem,
        df_aeroportos_anac_destino
    ])
    .filter(
        pl.col("municipio").is_not_null()
        & pl.col("codigo_icao").is_not_null()
    )
    .unique()
    .collect()
)

df_aeroportos_anac.head()

In [ ]:
# ============================================================
# 7.3 DIMENSÃO FINAL DE AEROPORTOS
# ============================================================

dim_aeroportos = (
    df_aeroportos_anac
    .join(
        df_aeroportos,
        on="codigo_icao",
        how="inner"
    )
    .with_columns([
        pl.col("codigo_iata").alias("iata"),

        expr_texto_normalizado(
            "municipio"
        ).alias("municipio_chave")
    ])
    .select([
        "municipio",
        "municipio_chave",
        "codigo_icao",
        "iata",
        "nome_aeroporto"
    ])
    .unique()
    .sort([
        "municipio",
        "iata"
    ])
)

print(dim_aeroportos.shape)

dim_aeroportos.head(20)

In [ ]:
df_aeroportos_anac.filter(
    pl.col("municipio").str.to_lowercase().str.contains("guarulhos")
)

In [ ]:
df_aeroportos.filter(
    pl.col("codigo_icao") == "SBGR"
)

In [ ]:
dim_aeroportos.filter(
    pl.col("codigo_icao") == "SBGR"
)

In [ ]:
# ============================================================
# 7.4 RESOLUÇÃO DE CIDADE OU IATA
# ============================================================

def resolver_local_para_iatas(local):

    local = str(local).strip()

    if not local:
        raise ValueError("Local não informado.")

    # --------------------------------------------------------
    # 1. Caso seja informado diretamente um IATA
    # --------------------------------------------------------

    if len(local) == 3 and local.isalpha():

        codigo = local.upper()

        encontrados = (
            dim_aeroportos
            .filter(
                pl.col("iata") == codigo
            )
            .select("iata")
            .unique()
            .to_series()
            .to_list()
        )

        if encontrados:
            return encontrados

    chave = normalizar_texto(local)

    # --------------------------------------------------------
    # 2. Busca exata pelo município
    # --------------------------------------------------------

    encontrados = (
        dim_aeroportos
        .filter(
            pl.col("municipio_chave") == chave
        )
        .select("iata")
        .unique()
        .sort("iata")
        .to_series()
        .to_list()
    )

    if encontrados:
        return encontrados

    # --------------------------------------------------------
    # 3. Busca parcial pelo município
    # --------------------------------------------------------

    encontrados = (
        dim_aeroportos
        .filter(
            pl.col("municipio_chave")
            .str.contains(chave)
        )
        .select("iata")
        .unique()
        .sort("iata")
        .to_series()
        .to_list()
    )

    if encontrados:
        return encontrados

    # --------------------------------------------------------
    # 4. Busca pelo nome do aeroporto
    # Ex.: "Guarulhos" encontra
    # "São Paulo/Guarulhos ... International Airport"
    # --------------------------------------------------------

    encontrados = (
        dim_aeroportos
        .with_columns(
            expr_texto_normalizado(
                "nome_aeroporto"
            ).alias("nome_aeroporto_chave")
        )
        .filter(
            pl.col("nome_aeroporto_chave")
            .str.contains(chave)
        )
        .select("iata")
        .unique()
        .sort("iata")
        .to_series()
        .to_list()
    )

    if encontrados:
        return encontrados

    raise ValueError(
        f"Nenhum aeroporto encontrado para '{local}'."
    )

## 8. Consulta ao Google Flights/SerpApi

A consulta mantém a lógica do rascunho original, mas:

- aceita vários aeroportos em uma única requisição;
- captura `best_flights` e `other_flights`;
- identifica o aeroporto efetivamente utilizado;
- preserva companhias e números de voo;
- informa conexões.

Uma simulação do usuário corresponde a uma chamada à API, mesmo quando uma cidade possui vários aeroportos cadastrados.


In [ ]:
# ============================================================
# 8.1 FUNÇÃO DE COTAÇÃO
# ============================================================

def cotar_google_flights(
    iatas_origem,
    iatas_destino,
    data_voo,
):
    origem_param = ",".join(sorted(set(iatas_origem)))
    destino_param = ",".join(sorted(set(iatas_destino)))

    params = {
        "engine": "google_flights",
        "departure_id": origem_param,
        "arrival_id": destino_param,
        "outbound_date": data_voo.isoformat(),
        "currency": "BRL",
        "hl": "pt-br",
        "gl": "br",
        "type": "2",
        "show_hidden": "true",
        "api_key": API_KEY,
    }

    try:
        with httpx.Client(timeout=60.0) as client:
            response = client.get(
                URL_SERPAPI,
                params=params,
            )
            response.raise_for_status()
            results = response.json()

    except httpx.HTTPStatusError as erro:
        raise RuntimeError(
            f"Erro HTTP {erro.response.status_code}: "
            f"{erro.response.text}"
        ) from erro

    except Exception as erro:
        raise RuntimeError(
            f"Erro ao consultar SerpApi: {erro}"
        ) from erro

    if "error" in results:
        raise RuntimeError(
            f"Erro retornado pela SerpApi: {results['error']}"
        )

    ofertas = []

    for categoria in ["best_flights", "other_flights"]:
        for oferta in results.get(categoria, []):
            ofertas.append((categoria, oferta))

    if not ofertas:
        return pl.DataFrame()

    linhas = []
    momento_extracao = datetime.now()

    for categoria, oferta in ofertas:
        pernas = oferta.get("flights", [])

        if not pernas:
            continue

        primeiro = pernas[0]
        ultimo = pernas[-1]

        aeroporto_partida = primeiro.get(
            "departure_airport",
            {},
        )

        aeroporto_chegada = ultimo.get(
            "arrival_airport",
            {},
        )

        companhias = []
        numeros_voos = []

        for perna in pernas:
            companhia = perna.get("airline")

            if companhia and companhia not in companhias:
                companhias.append(companhia)

            numero = perna.get("flight_number")

            if numero:
                numeros_voos.append(str(numero))

        linhas.append({
            "fonte": "GoogleFlights_SerpApi",
            "categoria_resultado": categoria,
            "data_extracao": momento_extracao,
            "data_voo": data_voo,

            "iata_origem_consulta": origem_param,
            "iata_destino_consulta": destino_param,

            "iata_origem_voo": aeroporto_partida.get("id"),
            "aeroporto_origem_nome": aeroporto_partida.get("name"),

            "iata_destino_voo": aeroporto_chegada.get("id"),
            "aeroporto_destino_nome": aeroporto_chegada.get("name"),

            "companhia_principal_api": primeiro.get("airline"),
            "companhias_itinerario": " + ".join(companhias),

            "numeros_voos": " | ".join(numeros_voos),

            "partida_horario": aeroporto_partida.get("time"),
            "chegada_horario": aeroporto_chegada.get("time"),

            "duracao_minutos": oferta.get("total_duration"),
            "conexoes": max(len(pernas) - 1, 0),

            "preco_brl": oferta.get("price"),
            "tipo_tarifa": oferta.get("type"),
        })

    if not linhas:
        return pl.DataFrame()

    return (
        pl.DataFrame(linhas)
        .with_columns([
            pl.col("data_voo").cast(pl.Date),
            pl.col("preco_brl").cast(pl.Float64, strict=False),
            pl.col("duracao_minutos").cast(pl.Int32, strict=False),
            pl.col("conexoes").cast(pl.Int32, strict=False),
        ])
        .filter(pl.col("preco_brl").is_not_null())
        .unique(
            subset=[
                "data_voo",
                "iata_origem_voo",
                "iata_destino_voo",
                "companhia_principal_api",
                "partida_horario",
                "chegada_horario",
                "preco_brl",
            ],
            keep="first",
        )
        .sort([
            "preco_brl",
            "duracao_minutos",
        ])
    )


## 9. Preparação da consulta para a Gold

O aeroporto efetivamente retornado pela API é associado novamente ao município da base ANAC.

Assim, o `JOIN` usa a rota efetivamente encontrada e não apenas o texto digitado pelo usuário.


In [ ]:
# ============================================================
# 9.1 ADIÇÃO DOS MUNICÍPIOS E CHAVES TÉCNICAS
# ============================================================

def preparar_api_para_join(df_api):
    if df_api.is_empty():
        return df_api

    dim_iata = (
        dim_aeroportos
        .select([
            "iata",
            "municipio",
        ])
        .unique(subset=["iata"])
    )

    origem = dim_iata.rename({
        "iata": "iata_origem_voo",
        "municipio": "municipio_origem_gold",
    })

    destino = dim_iata.rename({
        "iata": "iata_destino_voo",
        "municipio": "municipio_destino_gold",
    })

    return (
        df_api
        .join(
            origem,
            on="iata_origem_voo",
            how="left",
        )
        .join(
            destino,
            on="iata_destino_voo",
            how="left",
        )
        .with_columns([
            expr_texto_normalizado(
                "municipio_origem_gold"
            ).alias("municipio_origem_chave"),

            expr_texto_normalizado(
                "municipio_destino_gold"
            ).alias("municipio_destino_chave"),

            expr_companhia_normalizada(
                "companhia_principal_api"
            ).alias("companhia_chave"),
        ])
    )


In [ ]:
# ============================================================
# 9.2 RECORTE DA GOLD PARA A CONSULTA
# ============================================================

COLUNAS_GOLD_DESEJADAS = [
    "data_voo",
    "municipio_origem",
    "municipio_destino",
    "nome_empresa",

    "score_cancelamento",
    "score_atraso",
    "score_risco_operacional",

    "hist_empresa_pct_cancelamento",
    "hist_empresa_pct_atraso",

    "hist_rota_pct_cancelamento",
    "hist_rota_pct_atraso",

    "hist_empresa_rota_pct_cancelamento",
    "hist_empresa_rota_pct_atraso",

    "hist_data_pct_cancelamento",
    "hist_data_pct_atraso",

    "nivel_referencia_preco",
    "preco_estimado_modelo",
    "preco_historico_media",
    "preco_historico_mediana",
    "preco_historico_desvio",
    "preco_historico_q1",
    "preco_historico_q3",
    "preco_historico_iqr",
    "score_volatilidade_preco",
    "score_risco_base",

    "data_processamento",
    "versao_modelo",
]


def carregar_gold_consulta(data_voo):
    colunas_disponiveis = set(
        gold_lazy.collect_schema().names()
    )

    colunas = [
        coluna
        for coluna in COLUNAS_GOLD_DESEJADAS
        if coluna in colunas_disponiveis
    ]

    df = (
        gold_lazy
        .filter(
            pl.col("data_voo") == data_voo
        )
        .select(colunas)
        .collect()
    )

    if df.is_empty():
        return df

    return (
        df
        .with_columns([
            expr_texto_normalizado(
                "municipio_origem"
            ).alias("municipio_origem_chave"),

            expr_texto_normalizado(
                "municipio_destino"
            ).alias("municipio_destino_chave"),

            expr_companhia_normalizada(
                "nome_empresa"
            ).alias("companhia_chave"),
        ])
    )


## 10. Score da tarifa observada

Com o preço real retornado pela API, são calculados três componentes:

1. **IQR**: distância acima do terceiro quartil;
2. **desvio padrão**: distância acima da média histórica;
3. **modelo de preço**: excesso em relação ao preço previsto.

Cada componente é convertido para a escala `0–1`.

- próximo de `0`: tarifa compatível ou favorável;
- próximo de `1`: tarifa muito acima das referências.

O `score_risco_geral` combina cancelamento, atraso e preço atual.


In [ ]:
# ============================================================
# 10.1 CÁLCULO DO SCORE DE PREÇO ATUAL
# ============================================================

def adicionar_score_preco(df):
    if df.is_empty():
        return df

    colunas_necessarias = [
        "preco_historico_iqr",
        "preco_historico_q3",
        "preco_historico_desvio",
        "preco_historico_media",
        "preco_estimado_modelo",
        "preco_historico_q1",
    ]

    for coluna in colunas_necessarias:
        if coluna not in df.columns:
            df = df.with_columns(
                pl.lit(None, dtype=pl.Float64).alias(coluna)
            )

    df = (
        df
        .with_columns([
            pl.when(
                (pl.col("preco_historico_iqr") > 0)
                & pl.col("preco_historico_q3").is_not_null()
            )
            .then(
                (
                    (
                        pl.col("preco_brl")
                        - pl.col("preco_historico_q3")
                    )
                    / pl.col("preco_historico_iqr")
                )
                .clip(lower_bound=0)
            )
            .otherwise(None)
            .alias("_excesso_iqr"),

            pl.when(
                (pl.col("preco_historico_desvio") > 0)
                & pl.col("preco_historico_media").is_not_null()
            )
            .then(
                (
                    (
                        pl.col("preco_brl")
                        - pl.col("preco_historico_media")
                    )
                    / pl.col("preco_historico_desvio")
                )
                .clip(lower_bound=0)
            )
            .otherwise(None)
            .alias("_z_preco"),

            pl.when(
                pl.col("preco_estimado_modelo") > 0
            )
            .then(
                (
                    (
                        pl.col("preco_brl")
                        / pl.col("preco_estimado_modelo")
                    )
                    - 1
                )
                .clip(lower_bound=0)
            )
            .otherwise(None)
            .alias("_excesso_modelo"),
        ])
        .with_columns([
            (
                1 - (-pl.col("_excesso_iqr")).exp()
            )
            .clip(0, 1)
            .alias("score_preco_iqr"),

            (
                1 - (-pl.col("_z_preco")).exp()
            )
            .clip(0, 1)
            .alias("score_preco_desvio"),

            (
                1
                - (
                    -2 * pl.col("_excesso_modelo")
                ).exp()
            )
            .clip(0, 1)
            .alias("score_preco_modelo"),
        ])
    )

    componentes = [
        "score_preco_iqr",
        "score_preco_desvio",
        "score_preco_modelo",
    ]

    soma = pl.sum_horizontal([
        pl.col(c).fill_null(0)
        for c in componentes
    ])

    qtd = pl.sum_horizontal([
        pl.col(c).is_not_null().cast(pl.Int8)
        for c in componentes
    ])

    return (
        df
        .with_columns(
            pl.when(qtd > 0)
            .then(soma / qtd)
            .otherwise(None)
            .alias("score_preco_atual")
        )
        .with_columns(
            pl.when(
                pl.col("preco_historico_q1").is_not_null()
                & (
                    pl.col("preco_brl")
                    <= pl.col("preco_historico_q1")
                )
            )
            .then(pl.lit("favoravel"))

            .when(
                pl.col("preco_historico_q3").is_not_null()
                & (
                    pl.col("preco_brl")
                    <= pl.col("preco_historico_q3")
                )
            )
            .then(pl.lit("dentro_do_historico"))

            .when(
                pl.col("preco_historico_iqr").is_not_null()
                & (
                    pl.col("preco_brl")
                    <= (
                        pl.col("preco_historico_q3")
                        + 1.5 * pl.col("preco_historico_iqr")
                    )
                )
            )
            .then(pl.lit("acima_do_historico"))

            .when(
                pl.col("preco_historico_q3").is_not_null()
            )
            .then(pl.lit("muito_acima_do_historico"))

            .otherwise(pl.lit("sem_referencia"))
            .alias("classificacao_preco")
        )
        .drop([
            "_excesso_iqr",
            "_z_preco",
            "_excesso_modelo",
        ])
    )


In [ ]:
# ============================================================
# 10.2 SCORE DE RISCO GERAL
# ============================================================

def adicionar_score_risco_geral(df):
    if df.is_empty():
        return df

    for coluna in [
        "score_cancelamento",
        "score_atraso",
        "score_preco_atual",
    ]:
        if coluna not in df.columns:
            df = df.with_columns(
                pl.lit(None, dtype=pl.Float64).alias(coluna)
            )

    componentes = [
        "score_cancelamento",
        "score_atraso",
        "score_preco_atual",
    ]

    soma = pl.sum_horizontal([
        pl.col(c).fill_null(0)
        for c in componentes
    ])

    qtd = pl.sum_horizontal([
        pl.col(c).is_not_null().cast(pl.Int8)
        for c in componentes
    ])

    return (
        df
        .with_columns(
            pl.when(qtd > 0)
            .then(soma / qtd)
            .otherwise(None)
            .alias("score_risco_geral")
        )
        .with_columns(
            pl.when(
                pl.col("score_risco_geral") < 0.25
            )
            .then(pl.lit("baixo"))
            .when(
                pl.col("score_risco_geral") < 0.50
            )
            .then(pl.lit("moderado"))
            .when(
                pl.col("score_risco_geral") < 0.75
            )
            .then(pl.lit("alto"))
            .when(
                pl.col("score_risco_geral").is_not_null()
            )
            .then(pl.lit("muito_alto"))
            .otherwise(pl.lit("sem_score"))
            .alias("classificacao_risco_geral")
        )
    )


# 11. Função principal do simulador

Fluxo:

`input → IATAs → SerpApi → voos reais → Gold → scores`


In [ ]:
# ============================================================
# 11.1 FUNÇÃO PRINCIPAL
# ============================================================

def simular_viagem(
    cidade_origem,
    cidade_destino,
    data_viagem,
):
    if isinstance(data_viagem, str):
        valor = data_viagem.strip()

        formatos = [
            "%Y-%m-%d",
            "%d/%m/%Y",
        ]

        data_convertida = None

        for formato in formatos:
            try:
                data_convertida = datetime.strptime(
                    valor,
                    formato,
                ).date()
                break
            except ValueError:
                pass

        if data_convertida is None:
            raise ValueError(
                "Data inválida. Use DD/MM/AAAA ou AAAA-MM-DD."
            )

        data_viagem = data_convertida

    if not isinstance(data_viagem, date):
        raise TypeError("data_viagem deve ser date ou string.")

    if (
        data_viagem < DATA_MIN_GOLD
        or data_viagem > DATA_MAX_GOLD
    ):
        raise ValueError(
            f"A data {data_viagem} está fora do período "
            f"materializado na Gold "
            f"({DATA_MIN_GOLD} a {DATA_MAX_GOLD})."
        )

    iatas_origem = resolver_local_para_iatas(
        cidade_origem
    )

    iatas_destino = resolver_local_para_iatas(
        cidade_destino
    )

    print(
        "Origem:",
        cidade_origem,
        "=>",
        ", ".join(iatas_origem),
    )

    print(
        "Destino:",
        cidade_destino,
        "=>",
        ", ".join(iatas_destino),
    )

    print("Data:", data_viagem)
    print("Realizando 1 consulta à SerpApi...")

    df_api = cotar_google_flights(
        iatas_origem=iatas_origem,
        iatas_destino=iatas_destino,
        data_voo=data_viagem,
    )

    if df_api.is_empty():
        print("Nenhum voo foi retornado pela API.")
        return df_api

    print(
        f"{df_api.height} ofertas retornadas pela API."
    )

    df_api = preparar_api_para_join(df_api)

    df_gold_dia = carregar_gold_consulta(
        data_viagem
    )

    if df_gold_dia.is_empty():
        print(
            "Não existem registros na Gold para a data consultada."
        )

        return df_api

    resultado = (
        df_api
        .join(
            df_gold_dia,
            on=[
                "data_voo",
                "municipio_origem_chave",
                "municipio_destino_chave",
                "companhia_chave",
            ],
            how="left",
            suffix="_gold",
        )
        .with_columns(
            pl.when(
                pl.col("score_cancelamento").is_not_null()
            )
            .then(pl.lit("match_exato"))
            .otherwise(pl.lit("sem_match_exato"))
            .alias("status_match_gold")
        )
    )

    resultado = adicionar_score_preco(
        resultado
    )

    resultado = adicionar_score_risco_geral(
        resultado
    )

    return resultado.sort([
        "preco_brl",
        "score_risco_geral",
    ])


# 12. Input do usuário

Exemplo:

```text
Cidade de origem: São Paulo
Cidade de destino: Recife
Data: 20/12/2026
```

Também é possível informar diretamente IATAs, como `CGH` e `REC`.


In [ ]:
# ============================================================
# 12.1 ENTRADA DO USUÁRIO
# ============================================================

cidade_origem = input(
    "Cidade de origem ou IATA: "
).strip()

cidade_destino = input(
    "Cidade de destino ou IATA: "
).strip()

data_viagem = input(
    "Data da viagem (DD/MM/AAAA): "
).strip()


In [ ]:
# ============================================================
# 12.2 EXECUÇÃO DA SIMULAÇÃO
# ============================================================

df_resultado = simular_viagem(
    cidade_origem=cidade_origem,
    cidade_destino=cidade_destino,
    data_viagem=data_viagem,
)

print(df_resultado.shape)

df_resultado.head(100)


In [ ]:
df_resultado.filter(pl.col('companhia_principal_api')=='Azul')

## 13. Visão simplificada para o usuário

A tabela completa permanece disponível em `df_resultado`. A visão abaixo mantém os campos mais úteis para a decisão.


In [ ]:
# ============================================================
# 13.1 RESULTADO RESUMIDO
# ============================================================

COLUNAS_RESUMO = [
    "companhia_principal_api",
    "companhias_itinerario",

    "iata_origem_voo",
    "iata_destino_voo",
    "municipio_origem_gold",
    "municipio_destino_gold",

    "data_voo",
    "partida_horario",
    "chegada_horario",

    "duracao_minutos",
    "conexoes",
    "preco_brl",

    "score_cancelamento",
    "score_atraso",

    "preco_estimado_modelo",
    "preco_historico_mediana",
    "preco_historico_q1",
    "preco_historico_q3",

    "score_preco_atual",
    "classificacao_preco",

    "score_risco_geral",
    "classificacao_risco_geral",

    "status_match_gold",
]

colunas_existentes = [
    coluna
    for coluna in COLUNAS_RESUMO
    if coluna in df_resultado.columns
]

df_resultado_resumo = (
    df_resultado
    .select(colunas_existentes)
    .sort("preco_brl"))

df_resultado_resumo


In [ ]:
df_resultado_resumo.filter(pl.col('status_match_gold')!='sem_match_exato')

In [ ]:
# ============================================================
# 14. DIAGNÓSTICO DE MATCH COM A GOLD
# ============================================================

if not df_resultado.is_empty():
    diagnostico = (
        df_resultado
        .group_by([
            "companhia_principal_api",
            "companhia_chave",
            "status_match_gold",
        ])
        .agg(
            pl.len().alias("qtd_ofertas")
        )
        .sort([
            "status_match_gold",
            "companhia_principal_api",
        ])
    )

    diagnostico


## 15. Observação sobre voos com conexão

A Gold foi construída na granularidade:

`município de origem × município de destino × companhia × data`

Para voos diretos, o `JOIN` representa diretamente a rota histórica.

Em itinerários com conexão ou mais de uma companhia, o notebook mantém a oferta, mas associa o score à **companhia principal** e à rota completa. Se essa combinação não existir na Gold, `status_match_gold` será `sem_match_exato`, evitando atribuir artificialmente um score de outra rota.


In [ ]:
# ============================================================
# 16. MATERIALIZAR A CONSULTA
# ============================================================

pasta_consultas = (f"{PASTA_SPEC}/spec_resultados/consultas_google_flights")
os.makedirs(
     pasta_consultas,
     exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
arquivo_consulta = (f"{pasta_consultas}/consulta_voos_{timestamp}.parquet")
df_resultado.write_parquet(arquivo_consulta,compression="zstd",)

print(arquivo_consulta)
